In [7]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib as plt

os.chdir("/Users/annacarolinafagundes/ibd-myeloid-scoring/")
sc.settings.figdir = "figures/scrna/"
sc.settings.verbosity = 1

In [ ]:
#load object
adata_myeloid=sc.read_h5ad("data/processed/scrna/adata_myeloid_annotated.h5ad")


## Extract gene signatures from Chron's disease group

In [ ]:
#subset CD group
adata_cd=adata_myeloid[adata_myeloid.obs["condition"]=="CD"].copy()
adata_cd.obs

sc.tl.rank_genes_groups(
    adata=adata_cd, 
    groupby="cell_type",
    reference='rest',
    method="wilcoxon",
    layer="counts", #raw data
    pts=True #pct
    )

#extract results
deg_results={}
for ct in cell_types:
    result = sc.get.rank_genes_groups_df(adata_cd, group=ct)
    result["cell_type"] = ct
    deg_results[ct] = result
print("Done")


Done


In [ ]:
#extract top 100 genes per cluster
signatures = {}
for ct, df in deg_results.items():
    filtered = df[
        (df["logfoldchanges"]>1.0)&
        (df["pvals_adj"]<0.05)&
        (df["pct_nz_group"]>0.25)
    ].sort_values("logfoldchanges", ascending=False).head(100)

    signatures[ct]=filtered
    print(f"{ct}: 100 significant marker genes")

M0 Macrophages: 100 significant marker genes
Neutrophil 1: 100 significant marker genes
Neutrophil 2: 100 significant marker genes
M2.2 Macrophages: 100 significant marker genes
Mixed Macrophages: 100 significant marker genes
Neutrophil 3: 100 significant marker genes
DCs: 100 significant marker genes
M1 Macrophages: 100 significant marker genes
IDA Macrophages: 100 significant marker genes
M2 Macrophages: 100 significant marker genes
M0_Ribhi Macrophages: 100 significant marker genes
Inflammatory Monocytes: 100 significant marker genes


In [ ]:
# Combine all cell type results into one DataFrame
signatures_df = pd.concat(signatures.values(), ignore_index=True)

# Save to file
signatures_df.to_csv("data/processed/scrna/myeloid_cd_signatures.csv", index=False)
print(signatures_df.shape)
print(signatures_df["cell_type"].value_counts())

(1200, 8)
cell_type
M0 Macrophages            100
Neutrophil 1              100
Neutrophil 2              100
M2.2 Macrophages          100
Mixed Macrophages         100
Neutrophil 3              100
DCs                       100
M1 Macrophages            100
IDA Macrophages           100
M2 Macrophages            100
M0_Ribhi Macrophages      100
Inflammatory Monocytes    100
Name: count, dtype: int64


In [ ]:
# Save just gene names per cell type as a simple dictionary
import json
gene_lists = {ct: df["names"].tolist() for ct, df in signatures.items()}
with open("data/processed/scrna/myeloid_cd_gene_lists.json", "w") as f:
    json.dump(gene_lists, f, indent=2)
print("Saved gene lists")

Saved gene lists
